# Docling 结构感知切分

Docling 是由 IBM Research 创建、目前在 LF AI & Data Foundation 下维护的开源文档转换与版面分析项目。它与 MinerU 属于同一类上游文档解析工具：先从 PDF、DOCX、PPTX、HTML 或图片中恢复结构，再把结果交给 RAG。

Docling 的统一结构对象称为 `DoclingDocument`，可以表达文本、标题层级、表格、图片、列表、阅读顺序、页码、Bounding Box 和来源信息。

## 与 MinerU 的简单对比

| 方面 | Docling | MinerU |
| --- | --- | --- |
| 定位 | 文档转换、结构模型与原生 Chunker | PDF/图片解析、Markdown/JSON/资源输出 |
| 统一结构 | `DoclingDocument` Pydantic 模型 | MinerU 中间 JSON 与导出产物 |
| RAG 切分 | 内置 `HierarchicalChunker`、`HybridChunker` | 通常在解析输出后自行实现 |
| LangChain 接入 | 独立包 `langchain-docling` | 通常自行封装 Adapter |
| DualStruct 中的角色 | 可新增 Docling Parser Adapter | 当前使用 MinerU API + Normalizer |

两者不是 LangChain TextSplitter。它们位于更上游的“文档理解”层。

## Docling 的两种主要 Chunker

### HierarchicalChunker

按照 `DoclingDocument` 的元素和层级创建 Chunk，并附带标题、Caption 和原始元素引用等 metadata。

### HybridChunker

先使用层级切分结果，再根据目标 Tokenizer：

1. 拆分超过 Token 上限的 Chunk。
2. 合并标题与 Caption 相同、长度较短的相邻 Chunk。
3. 表格跨块时可以重复表头。

这正是“文档结构优先、Token 长度兜底”的实现。

## 可选代码示意

Docling 依赖较重，会间接安装 NumPy、Pandas、Transformers 等包，因此当前学习项目不将它设为必装依赖。需要单独实验时再执行 `uv add langchain-docling`。

```python
from docling.chunking import HybridChunker
from langchain_docling import DoclingLoader

loader = DoclingLoader(
    file_path="document.pdf",
    chunker=HybridChunker(
        tokenizer="sentence-transformers/all-MiniLM-L6-v2",
        max_tokens=512,
        merge_peers=True,
    ),
)
documents = loader.load()
```

`langchain-docling` 负责把 Docling Chunk 适配成 LangChain `Document`；真正的解析和结构切分由 Docling 完成。

## 官方资料

- [DoclingDocument](https://docling-project.github.io/docling/concepts/docling_document/)
- [Docling Chunking](https://docling-project.github.io/docling/concepts/chunking/)
- [LangChain Docling Loader](https://docs.langchain.com/oss/python/integrations/document_loaders/docling)